# RMS AI Vector Search Model Training

Use this notebook in Google Colab to train the RMS embedding model with GPU, export it to ONNX, and download the runtime model for `packages/ai-models/rms-embedding-model`.

Runtime requirements in this project:

- Base/default model: `intfloat/multilingual-e5-small`
- Embedding dimension: `384`
- Runtime format: Transformers.js local ONNX layout
- Final target folder in repo: `packages/ai-models/rms-embedding-model`


## 1. Select GPU

In Colab, open `Runtime > Change runtime type > T4 GPU` before running the notebook.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Load project source

Option A is recommended when the project is available on GitHub. Set `REPO_URL`, then run the cell.

If you open this notebook through a Colab Extension and the repo files are already present, set `PROJECT_DIR` to that folder instead.

In [ ]:
from pathlib import Path

REPO_URL = ""  # Example: "https://github.com/your-org/SE20A05Group7RMS.git"
PROJECT_DIR = Path("/content/SE20A05Group7RMS")

if REPO_URL and not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

if not PROJECT_DIR.exists():
    raise RuntimeError("Set REPO_URL or upload/open the project folder, then set PROJECT_DIR correctly.")

%cd {PROJECT_DIR}
!pwd
!ls ml/scripts

## 3. Install training dependencies

This installs the Python training stack used by `ml/scripts/train.py` and `ml/scripts/convert.py`.

In [ ]:
!python -m pip install -U pip
!pip install -r ml/requirements.txt

## 4. Prepare training data

The repo already contains sample files:

- `ml/data/raw/job_descriptions.csv`
- `ml/data/raw/cv_chunks.csv`
- `ml/data/triplets.train.jsonl`
- `ml/data/triplets.dev.jsonl`

For real training, replace the CSV files with stronger RMS data, then rebuild triplets.

In [ ]:
# Optional: upload replacement CSV files from your machine.
# They must be named job_descriptions.csv and cv_chunks.csv.
UPLOAD_NEW_CSV = False

if UPLOAD_NEW_CSV:
    from google.colab import files
    uploaded = files.upload()
    raw_dir = PROJECT_DIR / "ml" / "data" / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    for name, content in uploaded.items():
        target = raw_dir / name
        target.write_bytes(content)
        print("Uploaded", target)

In [ ]:
!python ml/scripts/build_triplets.py \
  --jobs ml/data/raw/job_descriptions.csv \
  --cvs ml/data/raw/cv_chunks.csv \
  --out ml/data/triplets.train.jsonl \
  --augment 2

!python - <<'PY'
from pathlib import Path
train = Path('ml/data/triplets.train.jsonl')
dev = Path('ml/data/triplets.dev.jsonl')
print('train rows:', sum(1 for _ in train.open(encoding='utf-8')))
print('dev rows:', sum(1 for _ in dev.open(encoding='utf-8')) if dev.exists() else 0)
PY

## 5. Train the embedding model

Keep `BASE_MODEL` 384-dimensional unless you also migrate the database `cv_embeddings.embedding vector(384)` schema.

In [ ]:
BASE_MODEL = "intfloat/multilingual-e5-small"
EPOCHS = 2
BATCH_SIZE = 16

!python ml/scripts/train.py \
  --base-model {BASE_MODEL} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --train ml/data/triplets.train.jsonl \
  --dev ml/data/triplets.dev.jsonl \
  --output ml/exported/rms-embedding-model \
  --checkpoints ml/checkpoints/rms-embedding-model

## 6. Export to ONNX runtime layout

The Node.js app loads this folder through `@wr/ai` with local Transformers.js.

In [ ]:
!python ml/scripts/convert.py \
  --model ml/exported/rms-embedding-model \
  --out packages/ai-models/rms-embedding-model

!find packages/ai-models/rms-embedding-model -maxdepth 3 -type f | sort

## 7. Validate embedding dimension

This confirms the trained model still outputs 384 dimensions for pgvector.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("ml/exported/rms-embedding-model")
embedding = model.encode(["query: Backend Developer with Node.js and PostgreSQL"], normalize_embeddings=True)
print("Embedding shape:", embedding.shape)
assert embedding.shape[-1] == 384, "Expected 384 dimensions"

## 8. Download model artifact

Download the zip, extract it locally, and place the extracted `rms-embedding-model` folder at:

`packages/ai-models/rms-embedding-model`

In [ ]:
!cd packages/ai-models && zip -r /content/rms-embedding-model.zip rms-embedding-model

from google.colab import files
files.download("/content/rms-embedding-model.zip")

## 9. Optional: save to Google Drive

Use this when you want to keep checkpoints and exported models after the Colab runtime shuts down.

In [ ]:
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/rms-ai-models
    !cp /content/rms-embedding-model.zip /content/drive/MyDrive/rms-ai-models/rms-embedding-model.zip
    !cp -r ml/checkpoints/rms-embedding-model /content/drive/MyDrive/rms-ai-models/checkpoints
    print("Saved to Google Drive: /content/drive/MyDrive/rms-ai-models")